# Proyecto completo de Machine Learning usando librerías
## Ejemplo didáctico: clasificación con Árbol de Decisión

Este notebook muestra, de manera básica y ordenada, las fases principales de un proyecto de Machine Learning usando librerías estándar de Python:
- `pandas` para manipulación de datos.
- `numpy` para operaciones numéricas.
- `matplotlib` y `seaborn` para visualización.
- `scikit-learn` para preprocesamiento, entrenamiento, validación, evaluación y afinamiento del modelo.


---

In [ ]:
# Importación de librerías básicas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración de estilo para gráficos
sns.set_theme(style="whitegrid")
import warnings
warnings.filterwarnings('ignore')


## 1. Carga del dataset
Para este ejemplo didáctico, usaremos el dataset **Breast Cancer Wisconsin (Diagnostic)**, disponible en `scikit-learn`. El objetivo es clasificar tumores en benignos o malignos.

In [ ]:
from sklearn.datasets import load_breast_cancer

# Cargar los datos
cancer_data = load_breast_cancer()
X = pd.DataFrame(cancer_data.data, columns=cancer_data.feature_names)
y = pd.Series(cancer_data.target, name='target') # 0: maligno, 1: benigno

# Unir en un solo DataFrame para el análisis exploratorio
df = pd.concat([X, y], axis=1)

# Visualizar las primeras filas
display(df.head())


## 2. Análisis Exploratorio de Datos (EDA)
Exploramos la distribución de las clases, estadísticas descriptivas y correlaciones principales.

In [ ]:
# Información general del dataset
print(df.info())

# Estadísticas descriptivas básicas
display(df.describe())

# Distribución de la variable objetivo (clases)
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='target', palette='Set2')
plt.title('Distribución de Clases (0: Maligno, 1: Benigno)')
plt.show()

# Correlación entre algunas características y la variable objetivo
# Mostramos el top 10 de características con mayor correlación absoluta con el target
correlaciones = df.corr()['target'].drop('target').sort_values()
top_corr = pd.concat([correlaciones.head(5), correlaciones.tail(5)])

plt.figure(figsize=(8, 5))
top_corr.plot(kind='barh', color='skyblue')
plt.title('Top 10 características más correlacionadas con el Target')
plt.xlabel('Coeficiente de Correlación de Pearson')
plt.show()


## 3. Limpieza de Datos
En datasets reales, este paso implica manejar valores nulos (`NaN`), duplicados o datos atípicos (outliers). 
En este dataset de scikit-learn, los datos ya vienen limpios, pero simularemos una revisión.

In [ ]:
# Verificar valores nulos
print("Valores nulos por columna (Top 5):")
print(df.isnull().sum().sort_values(ascending=False).head())

# Verificar filas duplicadas
print(f"\nFilas duplicadas: {df.duplicated().sum()}")

# Si hubiera nulos, podríamos usar df.dropna() o rellenarlos con df.fillna(df.median())


## 4. Ingeniería de Características (Feature Engineering)
Consiste en transformar características existentes o crear nuevas para ayudar al modelo.
Para árboles de decisión, el escalado (StandardScaler, MinMaxScaler) **no es estrictamente necesario**, pero aquí calcularemos una nueva característica de ejemplo.

In [ ]:
# Crear una nueva característica (ejemplo didáctico: ratio entre radio y textura media)
# Esto puede o no ayudar al modelo, pero ilustra cómo crear nuevas features.
X['mean_radius_texture_ratio'] = X['mean radius'] / (X['mean texture'] + 1e-5)

# Verificar la nueva característica
print(X[['mean radius', 'mean texture', 'mean_radius_texture_ratio']].head())


## 5. Preparación de datos para entrenamiento
Separamos los datos en un conjunto de entrenamiento (para que el modelo aprenda) y un conjunto de prueba (para evaluar su desempeño).

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2,       # 20% para prueba, 80% para entrenamiento
    random_state=42,     # Semilla para reproducibilidad
    stratify=y           # Mantiene la proporción de clases en train y test
)

print(f"Tamaño del set de entrenamiento: {X_train.shape}")
print(f"Tamaño del set de prueba: {X_test.shape}")


## 6. Selección del modelo y 7. Entrenamiento
Seleccionamos el **Árbol de Decisión (`DecisionTreeClassifier`)** y lo entrenamos con los datos de entrenamiento.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

# 6. Instanciar el modelo con hiperparámetros básicos
# max_depth limita la profundidad del árbol para evitar sobreajuste (overfitting)
clf = DecisionTreeClassifier(max_depth=4, random_state=42)

# 7. Entrenamiento (Ajuste del modelo)
clf.fit(X_train, y_train)

print("¡Modelo entrenado exitosamente!")


## 8. Evaluación
Evaluamos qué tan bien generaliza el modelo usando los datos de prueba (`X_test`, `y_test`).

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Predicciones sobre el set de prueba
y_pred = clf.predict(X_test)

# Métrica de Precisión Global (Accuracy)
acc = accuracy_score(y_test, y_pred)
print(f"Precisión (Accuracy): {acc:.4f}\n")

# Reporte de clasificación (Precisión, Recall, F1-Score)
print("Reporte de Clasificación:")
print(classification_report(y_test, y_pred, target_names=['Maligno (0)', 'Benigno (1)']))

# Matriz de Confusión
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Pred. Maligno', 'Pred. Benigno'],
            yticklabels=['Real Maligno', 'Real Benigno'])
plt.title('Matriz de Confusión')
plt.show()


## 9. Validación con K-Fold Cross Validation
Para estar seguros de que nuestro modelo no es bueno por pura suerte con la división de `train_test_split`, aplicamos validación cruzada. El dataset se divide en $K$ partes y el modelo se entrena/evalúa $K$ veces.

In [ ]:
from sklearn.model_selection import cross_val_score

# Aplicamos 5-Fold Cross Validation sobre todo el conjunto de entrenamiento
scores = cross_val_score(clf, X_train, y_train, cv=5, scoring='accuracy')

print(f"Resultados de los 5 folds: {scores}")
print(f"Precisión Promedio (Cross-Validation): {scores.mean():.4f} (+/- {scores.std():.4f})")


## 10. Afinamiento con Grid Search (Hyperparameter Tuning)
Buscamos la mejor combinación de hiperparámetros (profundidad, criterio, etc.) para optimizar el árbol.

In [ ]:
from sklearn.model_selection import GridSearchCV

# Definir el espacio de búsqueda
param_grid = {
    'criterion': ['gini', 'entropy'],
    'max_depth': [None, 3, 4, 5, 7, 10],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

# Configurar GridSearchCV
grid_search = GridSearchCV(
    estimator=DecisionTreeClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,               # 5-fold cross validation interno
    scoring='accuracy',
    n_jobs=-1           # Usa todos los procesadores disponibles
)

# Ejecutar la búsqueda
grid_search.fit(X_train, y_train)

print(f"Mejores hiperparámetros encontrados: {grid_search.best_params_}")
print(f"Mejor precisión obtenida durante CV: {grid_search.best_score_:.4f}")


## 11. Entrenamiento final del mejor modelo
Usamos el mejor modelo encontrado por Grid Search y lo evaluamos en el set de prueba que apartamos inicialmente.

In [ ]:
# Obtener el mejor modelo
best_clf = grid_search.best_estimator_

# Predecir y evaluar sobre el set de prueba
y_pred_best = best_clf.predict(X_test)
best_acc = accuracy_score(y_test, y_pred_best)

print(f"Precisión Final en Test con el mejor modelo: {best_acc:.4f}")


## 12. Interpretación básica del Árbol
Una de las grandes ventajas de los árboles de decisión es que son modelos de "caja blanca"; podemos visualizar exactamente qué reglas aprendió para clasificar.

In [ ]:
from sklearn.tree import plot_tree

# Graficamos el árbol (limitamos max_depth a 3 en el gráfico para que sea legible)
plt.figure(figsize=(20, 10))
plot_tree(best_clf, 
          feature_names=X.columns,  
          class_names=['Maligno', 'Benigno'],
          filled=True, 
          rounded=True, 
          max_depth=3,
          fontsize=10)
plt.title("Visualización del Árbol de Decisión (Truncado a profundidad 3)")
plt.show()


---
### Conclusión
Hemos pasado por el flujo completo de un proyecto básico de Machine Learning: desde la ingesta y exploración de datos, pasando por limpieza y separación, hasta entrenar, optimizar y visualizar un modelo predictivo.